[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Amaciasagro/GIT-RemoteSensing/blob/master/soils_v2.ipynb)

# 🌱 Soil Spatial Analysis Module 
**Author:** Ariel Macías | Agronomist · GIS & Remote Sensing Data Scientist

**EN:** This notebook performs an advanced spatial analysis of agricultural fields using official data from the USDA-NRCS (Soil Data Mart) and Google Earth Engine (GEE). It extracts, structures, and visualizes edaphic properties for precision agriculture diagnostics.
**ES:** Este módulo realiza un análisis espacial avanzado de lotes agrícolas utilizando datos oficiales de la USDA-NRCS y Google Earth Engine. Extrae, estructura y visualiza propiedades edáficas para el diagnóstico en agricultura de precisión.

### ⚙️ Workflow / Flujo de Trabajo
| Step | Description (EN / ES) |
|------|-----------------------|
| **0** | Configuration & Credentials / *Configuración y Credenciales* |
| **1** | GEE Initialization & AOI Definition / *Inicialización y Definición del Lote* |
| **2** | WFS Download & Textural Mapping / *Descarga WFS y Mapeo Textural* |
| **3** | Tabular SDA Query & Agronomic Report / *Consulta Tabular SDA y Reporte* |
| **4** | Granulometric Composition Charts / *Gráficos de Composición Granulométrica* |

> ⚠️ **Note:** USDA-NRCS data is restricted to the **United States** territory.

In [ ]:
# ============================================================\n
# STEP 0: SETUP & IMPORTS
# ============================================================\n
import os
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']
os.environ['MPLBACKEND'] = 'pdf'
import matplotlib
import matplotlib.colors
import matplotlib.pyplot as plt
import ee
import geemap
import folium 
import geopandas as gpd
import pandas as pd
import requests
import io
import urllib
import warnings
import ipywidgets as widgets
import plotly.graph_objects as go
import plotly.express as px
import zipfile
from plotly.subplots import make_subplots
from shapely.geometry import Polygon, mapping
from shapely.ops import unary_union
from IPython.display import display, HTML, FileLink
from contextlib import redirect_stdout
from folium.features import GeoJsonTooltip
import json
import tempfile
import os

warnings.filterwarnings('ignore', message='Unverified HTTPS request')

# --- CONFIGURATION VARIABLES ---
GEE_PROJECT  = 'my-project'      # <-- Replace with your GEE Project ID
CENTER_LAT   = 33.584              # Initial map latitude (Texas/US region)
CENTER_LON   = -101.845            # Initial map longitude
INITIAL_ZOOM = 12

print("✅ Setup complete. Environment ready.")

### 1. Area of Interest (AOI) Definition / Definición del Lote
**EN:** Authenticate Google Earth Engine. Use the drawing tool on the map to define the field boundaries, or upload a spatial file (`.shp` or `.geojson`).
**ES:** Autentica GEE. Utiliza la herramienta de dibujo para delimitar el lote, o sube un archivo espacial.

In [ ]:
# --- GEE Authentication ---
try:
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine initialized successfully.")
except Exception:
    print("🔑 Authenticating Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print("✅ Earth Engine initialized successfully.")

In [ ]:
# ============================================================\n
# STEP 1: GEE AUTH & AOI DEFINITION
# ============================================================\n


# --- Shared Global Variables ---
field_geom_ee  = None   # ee.Geometry object
field_geom_shp = None   # shapely geometry object

# --- Interactive Map Setup ---
interactive_map = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=INITIAL_ZOOM)
interactive_map.add_basemap('SATELLITE')

# --- UI Widgets: Drawing ---
btn_confirm_draw = widgets.Button(
    description='✅ Confirm Drawn Polygon',
    button_style='success',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_draw = widgets.Output()

def confirm_polygon(b):
    global field_geom_ee, field_geom_shp
    with out_draw:
        out_draw.clear_output()
        roi = interactive_map.user_roi
        if roi is None:
            print("⚠️ No polygon detected. Please draw one first.")
            return

        roi_info       = roi.getInfo()
        field_geom_ee  = ee.Geometry(roi_info.get('geometry', roi_info))
        coords         = field_geom_ee.coordinates().getInfo()[0]
        field_geom_shp = Polygon(coords)

        area_ha = field_geom_shp.area * 111320**2 / 10000
        print(f"✅ Polygon confirmed: {len(coords)-1} vertices")
        print(f"   Estimated Area: {area_ha:.2f} ha")

btn_confirm_draw.unobserve_all()
btn_confirm_draw.on_click(confirm_polygon)

btn_download_shp = widgets.Button(
    description='📥 Download Drawn Polygon (.shp)',
    button_style='primary',
    layout=widgets.Layout(width='260px', margin='4px'),
    disabled=False  # Disabled until polygon is confirmed
)
out_download = widgets.Output()

def download_shp(b):
    with out_download:
        out_download.clear_output()
        gdf = gpd.GeoDataFrame(
            [{'name': 'field'}],
            geometry=[field_geom_shp],
            crs='epsg:4326'
        )
        tmp_dir = tempfile.mkdtemp()
        shp_path = os.path.join(tmp_dir, 'field.shp')
        gdf.to_file(shp_path)
        
        # Zip all shapefile components
        zip_path = 'field_boundary.zip'
        with zipfile.ZipFile(zip_path, 'w') as z:
            for ext in ['shp', 'shx', 'dbf', 'prj']:
                fp = os.path.join(tmp_dir, f'field.{ext}')
                if os.path.exists(fp):
                    z.write(fp, f'field.{ext}')
        
        display(FileLink(zip_path, result_html_prefix='👉 '))

btn_download_shp.unobserve_all()
btn_download_shp.on_click(download_shp)

# --- UI Widgets: File Upload ---
uploader = widgets.FileUpload(
    description='📂 Upload Field (.shp / .geojson)',
    accept='.shp,.zip,.geojson',
    multiple=True,
    layout=widgets.Layout(width='260px', margin='4px')
)
btn_upload = widgets.Button(
    description='📌 Load File to Map',
    button_style='info',
    layout=widgets.Layout(width='260px', margin='4px')
)
out_upload = widgets.Output()

def load_spatial_file(b):
    global field_geom_ee, field_geom_shp
    with out_upload:
        out_upload.clear_output()
        files = uploader.value

        if not files:
            print("⚠️ No file uploaded.")
            return

        tmp_dir = tempfile.mkdtemp()
        
        for file in files:
            name, content = file['name'], file['content']
            with open(os.path.join(tmp_dir, name), 'wb') as f:
                f.write(content)

        try:
            # File handling logic (extract zip if needed)
            zip_files = [f for f in os.listdir(tmp_dir) if f.endswith('.zip')]
            if zip_files:
                import zipfile
                with zipfile.ZipFile(os.path.join(tmp_dir, zip_files[0]), 'r') as z:
                    z.extractall(tmp_dir)
            
            valid_files = [f for f in os.listdir(tmp_dir) if f.endswith(('.shp', '.geojson'))]
            if not valid_files:
                print("❌ No valid spatial file found.")
                return

            gdf = gpd.read_file(os.path.join(tmp_dir, valid_files[0])).to_crs(epsg=4326)
            
            field_geom_shp = unary_union(gdf.geometry)
            field_geom_ee  = ee.Geometry(mapping(field_geom_shp))
            
            area_ha = field_geom_shp.area * 111320**2 / 10000
            area_ac = area_ha * 2.47105
            print(f"✅ File loaded. Estimated Area: {area_ha:.2f} ha, {area_ac:.2f} ac.")

            with redirect_stdout(io.StringIO()):
                interactive_map.add_gdf(gdf, layer_name="Uploaded Field", style={'color': '#f1c40f', 'fillOpacity': 0.2})
        except Exception as e:
            print(f"❌ Error processing file: {e}")

btn_upload.unobserve_all()
btn_upload.on_click(load_spatial_file)

btn_download_shp.disabled = False

# --- Render UI ---
display(widgets.HTML("<h3 style='margin:8px 0'>📍 AOI Definition Panel</h3>"))
display(interactive_map)
display(widgets.HBox([
    widgets.VBox([widgets.HTML("<b>Option A: Draw</b>"), btn_confirm_draw, out_draw,btn_download_shp, out_download]),
    widgets.VBox([widgets.HTML("<b>Option B: Upload</b>"), uploader, btn_upload, out_upload])
]))

### 2. Spatial WFS Query & Textural Mapping / Consulta WFS y Mapa Textural
**EN:** Connects to the USDA Soil Data Mart WFS service to retrieve spatial mapping units intersecting the AOI. Renders a choropleth map based on dominant soil texture.
**ES:** Se conecta al servicio WFS de USDA para descargar las unidades cartográficas. Genera un mapa coloreado según la textura dominante del suelo.

In [ ]:
# ============================================================\n
# STEP 2: WFS DOWNLOAD & CHOROPLETH MAP
# ============================================================\n

if field_geom_ee is None:
    raise ValueError("⚠️ AOI not defined. Please draw or upload a field in Step 1.")

minx, miny, maxx, maxy = field_geom_shp.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}"
centroid = [field_geom_shp.centroid.y, field_geom_shp.centroid.x]

# 2. WFS Request
wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params  = {
    'SERVICE': 'WFS', 'VERSION': '1.0.0', 'REQUEST': 'GetFeature',
    'TYPENAME': 'MapunitPoly', 'BBOX': bbox_str, 'SRSNAME': 'EPSG:4326', 'OUTPUTFORMAT': 'GML3'
}

print("📡 Connecting to USDA Soil Data Mart (WFS)...")
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError("❌ USDA WFS Connection failed.")

gdf_soils = gpd.read_file(io.BytesIO(response.content))
if gdf_soils.empty:
    raise ValueError("⚠️ No soil data found in this area (US coverage only).")

gdf_field   = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[field_geom_shp])
gdf_clipped = gpd.overlay(gdf_soils, gdf_field, how='intersection')
print(f"✅ {len(gdf_clipped)} soil units retrieved and clipped to AOI.")

# 3. Tabular Query for Dominant Texture
mukeys = tuple(int(k) for k in gdf_clipped['mukey'].unique())
mukeys_str = str(mukeys).replace(',)', ')') if len(mukeys) > 1 else f"({mukeys[0]})"

sql_texture = f"""
SELECT c.mukey, c.comppct_r, ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str} AND c.majcompflag = 'Yes'
AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

def get_texture_class(sand, silt, clay):
    if any(pd.isna([sand, silt, clay])): return 'No Data'
    s, si, c = float(sand), float(silt), float(clay)
    if c >= 40 and s <= 45 and si <= 40: return 'Clay'
    if c >= 40 and si >= 40:             return 'Silty Clay'
    if c >= 35 and s >= 45:              return 'Sandy Clay'
    if c >= 27 and s <= 45 and si >= 28: return 'Clay Loam'
    if c >= 27 and s >= 45:              return 'Sandy Clay Loam'
    if c >= 27 and si >= 60:             return 'Silty Clay Loam'
    if si >= 50 and c < 27:              return 'Silt Loam'
    if si >= 80 and c < 12:              return 'Silt'
    if s >= 85 and c < 10:               return 'Sand'
    if s >= 70 and c < 15:               return 'Loamy Sand'
    return 'Loam'

TEXTURE_PALETTE = {
    'Clay': '#8B0000', 'Silty Clay': '#B22222', 'Sandy Clay': '#CD5C5C',
    'Clay Loam': '#D2691E', 'Sandy Clay Loam': '#A0522D', 'Silty Clay Loam': '#C68642',
    'Silt Loam': '#DEB887', 'Silt': '#F5DEB3', 'Loam': '#8FBC8F',
    'Loamy Sand': '#F4A460', 'Sand': '#EDC9Af', 'No Data': '#AAAAAA'
}

# Fetch textures
df_texture = pd.DataFrame()
try:
    res = requests.post("https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
                        data={'query': sql_texture, 'format': 'JSON'}, verify=False)
    if res.status_code == 200 and 'Table' in res.json():
        df_texture = pd.DataFrame(res.json()['Table'], columns=['mukey','pct','sand','silt','clay'])
        df_texture['texture'] = df_texture.apply(lambda row: get_texture_class(row['sand'], row['silt'], row['clay']), axis=1)
except Exception as e:
    print(f"⚠️ Texture query failed: {e}")

# 4. Render Choropleth Map
soil_map = geemap.Map(center=centroid, zoom=11)     
soil_map.add_basemap('SATELLITE')

with redirect_stdout(io.StringIO()):
    soil_map.add_gdf(gdf_field, layer_name='AOI Boundary',
                     style={'color': 'white', 'weight': 2.5, 'fillOpacity': 0})
    
gdf_render = gdf_clipped[['mukey', 'musym', 'geometry']].copy()
if not df_texture.empty:
    dom_tex = df_texture.sort_values('pct', ascending=False).drop_duplicates('mukey')[['mukey','texture']]
    gdf_render['mukey'] = gdf_render['mukey'].astype(str)
    dom_tex['mukey'] = dom_tex['mukey'].astype(str)
    gdf_render = gdf_render.merge(dom_tex, on='mukey', how='left').fillna({'texture': 'No Data'})
else:
    gdf_render['texture'] = 'No Data'

for texture, group in gdf_render.groupby('texture'):
    color = TEXTURE_PALETTE.get(texture, '#AAAAAA')
    # fillColor sets the polygon fill; color sets the border; both must be specified
    style = {'color': '#555', 'weight': 1, 'fillColor': color, 'fillOpacity': 0.65}
    hover = {'fillColor': color, 'fillOpacity': 0.9}
    with redirect_stdout(io.StringIO()):
        soil_map.add_gdf(group, layer_name=f"Texture: {texture}",
                         style=style, hover_style=hover)

display(soil_map)

### 3. Agronomic Tabular Report / Reporte Agronómico
**EN:** Queries the Soil Data Access (SDA) API to retrieve critical chemical and physical properties (pH, Organic Matter, CEC, AWC) for structural diagnosis.
**ES:** Consulta la API de SDA para extraer propiedades químicas y físicas críticas (pH, Materia Orgánica, CIC, Agua Disponible) para el diagnóstico estructural.

### 4. Granulometric Composition Analysis / Análisis Granulométrico
**EN:** Automated visualization of sand, silt, and clay proportions using Plotly.
**ES:** Visualización automatizada de las proporciones de arena, limo y arcilla utilizando Plotly.

In [ ]:
# # ============================================================
# # STEP 4: PLOTLY GRANULOMETRIC VISUALIZATIONS
# # ============================================================

# if not series_data.empty:
#     # EN: Filter rows with valid granulometric data / ES: Filtrar filas con datos válidos
#     plot_data = series_data.dropna(subset=['arena', 'limo', 'arcilla']).copy()
#     plot_data = plot_data.sort_values('serie')

#     if not plot_data.empty:
#         # EN: Create Stacked Bar Chart / ES: Crear gráfico de barras apiladas
#         fig = go.Figure()

#         fig.add_trace(go.Bar(
#             y=plot_data['serie'], x=plot_data['arena'],
#             name='Sand / Arena (%)', orientation='h', marker=dict(color='#EDC9Af')
#         ))
#         fig.add_trace(go.Bar(
#             y=plot_data['serie'], x=plot_data['limo'],
#             name='Silt / Limo (%)', orientation='h', marker=dict(color='#F5DEB3')
#         ))
#         fig.add_trace(go.Bar(
#             y=plot_data['serie'], x=plot_data['arcilla'],
#             name='Clay / Arcilla (%)', orientation='h', marker=dict(color='#8B0000')
#         ))

#         fig.update_layout(
#             title="Granulometric Composition per Soil Series",
#             barmode='stack',
#             xaxis_title="Composition Percentage (%)",
#             yaxis_title="Soil Series",
#             plot_bgcolor='white',
#             height=400 + (len(plot_data) * 20),
#             legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
#         )

#         fig.show()
#     else:
#         print("⚠️ Not enough continuous numerical data to plot granulometry.")
# else:
#     print("⚠️ No data available to generate charts.")

In [ ]:
# ============================================================
# EN: CELL 3+4 — FULL HORIZON QUERY + WEIGHTED AVERAGES
# ES: CELDA 3+4 — CONSULTA COMPLETA DE HORIZONTES + PROMEDIOS PONDERADOS
# Interactive table by MUKey with critical value alerts
# ============================================================
import math
import numpy as np

# ⚠️ EN: Make sure these constants are defined in your Cell 0 / ES: Asegurate de definir estas constantes en tu Celda 0
MAX_DEPTH_CM = 80  # Reemplaza a PROF_MAX_CM
N_FACTOR = 0.05    # Reemplaza a FACTOR_N

# ── EN: Main SQL: all horizons + properties / ES: SQL principal ────
sql_horizons = (
    "SELECT\n"
    "    c.mukey, c.cokey, c.compname, c.comppct_r, c.majcompflag,\n"
    "    ch.chkey, ch.hzname, ch.hzdept_r, ch.hzdepb_r,\n"
    "    ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r,\n"
    "    ch.dbovendry_r,\n"
    "    ch.wtenthbar_r, ch.wthirdbar_r, ch.wfifteenbar_r,\n"
    "    ch.om_r,\n"
    "    ch.ph1to1h2o_r,\n"
    "    ch.ec_r, ch.ecec_r,\n"
    "    ch.esp_r, ch.sar_r,\n"
    "    ch.cec7_r\n"
    "FROM component AS c\n"
    "LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey\n"
    f"WHERE c.mukey IN {mukeys_str}\n"
    "  AND ch.hzdept_r IS NOT NULL\n"
    "  AND ch.hzdepb_r IS NOT NULL\n"
    "ORDER BY c.mukey, c.comppct_r DESC, ch.hzdept_r ASC"
)

HORIZON_COLS = [
    'mukey', 'cokey', 'compname', 'comppct_r', 'majcompflag',
    'chkey', 'hzname', 'hzdept_r', 'hzdepb_r',
    'sandtotal_r', 'silttotal_r', 'claytotal_r',
    'dbovendry_r',
    'wtenthbar_r', 'wthirdbar_r', 'wfifteenbar_r',
    'om_r',
    'ph1to1h2o_r',
    'ec_r', 'ecec_r',   
    'esp_r', 'sar_r',
    'cec7_r'
]

NUMERIC_COLS = HORIZON_COLS[3:4] + HORIZON_COLS[7:]

df_horizons = pd.DataFrame()
try:
    print("📡 EN: Querying chorizon table / ES: Consultando tabla chorizon...")
    r = requests.post(
        "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest",
        data={'query': sql_horizons, 'format': 'JSON'},
        verify=False, timeout=60
    )
    if r.status_code == 200 and 'Table' in r.json():
        df_horizons = pd.DataFrame(r.json()['Table'], columns=HORIZON_COLS)
        for col in NUMERIC_COLS:
            df_horizons[col] = pd.to_numeric(df_horizons[col], errors='coerce')
        df_horizons['n_estimado_r'] = df_horizons['om_r'] * N_FACTOR
        df_horizons['thickness_cm'] = df_horizons['hzdepb_r'] - df_horizons['hzdept_r']
        print(f"✅ EN: {len(df_horizons)} horizons downloaded / ES: {len(df_horizons)} horizontes descargados.")
    else:
        print(f"⚠️ EN: SDA Server returned code / ES: El servidor SDA devolvió código: {r.status_code}")
except Exception as e:
    print(f"❌ EN: SDA Query Error / ES: Error al consultar SDA: {e}")
    
    
# ── EN: Helper Functions / ES: Funciones auxiliares ────────────────────────

def calc_weighted_average(df_comp, variable, max_depth=MAX_DEPTH_CM):
    """EN: Weighted avg by root zone depth / ES: Promedio ponderado por espesor en zona radicular."""
    df = df_comp.copy()
    df['top_eff'] = df['hzdept_r'].clip(lower=0, upper=max_depth)
    df['bot_eff'] = df['hzdepb_r'].clip(lower=0, upper=max_depth)
    df['esp_eff'] = df['bot_eff'] - df['top_eff']
    df = df[(df['esp_eff'] > 0) & df[variable].notna()].copy()
    if df.empty: return np.nan
    total_esp = df['esp_eff'].sum()
    return (df[variable] * df['esp_eff']).sum() / total_esp if total_esp > 0 else np.nan

def calc_wpavg_by_mukey(df_horizons, variable, max_depth=MAX_DEPTH_CM):
    """EN: Calculate wpavg by mukey using dominant component / ES: Calcula wpavg por mukey."""
    results = []
    for mukey, df_mu in df_horizons.groupby('mukey'):
        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_comp  = df_mu[df_mu['cokey'] == comp_dom]
        value    = calc_weighted_average(df_comp, variable, max_depth)
        results.append({'mukey': str(mukey), f'{variable}_wpavg': value})
    return pd.DataFrame(results)


MAP_VARIABLES = {
    'Clay / Arcilla (%)':             'claytotal_r',
    'Sand / Arena (%)':               'sandtotal_r',
    'Silt / Limo (%)':                'silttotal_r',
    'Bulk Dens / Dens. ap. (g/cm³)':  'dbovendry_r',
    'Water / Agua 10 kPa (%)':        'wtenthbar_r',
    'Water / Agua 33 kPa (%)':        'wthirdbar_r',
    'Water / Agua 1500 kPa (%)':      'wfifteenbar_r',
    'OM / Mat. Orgánica (%)':         'om_r',
    'pH (H₂O 1:1)':                   'ph1to1h2o_r',
    'EC / CE (dS/m)':                 'ec_r',
    'ECEC (cmol/kg)':                 'ecec_r',
    'ESP / PSI (%)':                  'esp_r',
    'SAR':                            'sar_r',
    'CEC / CIC (cmol/kg)':            'cec7_r',
    'Est. N / N estimado (%) ⚠️':     'n_estimado_r',
}

# ── EN: Calculate weighted averages / ES: Calcular promedios ponderados ────
gdf_dash = gdf_clipped.copy()
gdf_dash['mukey'] = gdf_dash['mukey'].astype(str)

if not df_horizons.empty:
    wpavg_frames = []
    for label, col in MAP_VARIABLES.items():
        if col in df_horizons.columns:
            wpavg_frames.append(calc_wpavg_by_mukey(df_horizons, col))
    df_wpavg = wpavg_frames[0].copy()
    for df_tmp in wpavg_frames[1:]:
        df_wpavg = df_wpavg.merge(df_tmp, on='mukey', how='outer')

    # EN: Enriched GeoDataFrame / ES: GeoDataFrame enriquecido
    gdf_dash = gdf_dash.merge(df_wpavg, on='mukey', how='left')
    print(f"✅ EN: Weighted averages calculated / ES: Promedios ponderados calculados (0–{MAX_DEPTH_CM} cm).")

# ── EN: Visual Alert Functions / ES: Funciones de alerta visual ─────────────

def get_alert_style(col, val):
    """EN: Returns CSS styles based on alert thresholds / ES: Devuelve estilos CSS."""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return 'color:#999; text-align:center;'

    CRITICAL = 'background:#ffe4e1; color:#8b0000; font-weight:600; text-align:center; border-radius:3px;'
    WARNING  = 'background:#fff9e6; color:#b8860b; font-weight:600; text-align:center; border-radius:3px;'
    GOOD     = 'background:#f0f9f0; color:#2d6a2d; text-align:center;'
    NORMAL   = 'text-align:center; color:#333;'

    v = float(val)
    if col == 'ec_r':        return CRITICAL if v >= 4 else (WARNING if v >= 2 else NORMAL)
    if col == 'esp_r':       return CRITICAL if v >= 15 else (WARNING if v >= 6 else NORMAL)
    if col == 'sar_r':       return CRITICAL if v >= 13 else (WARNING if v >= 9 else NORMAL)
    if col in ('ph1to1h2o_r', 'phsaturated_r'):
        if v <= 4.5 or v >= 9.0: return CRITICAL
        if v <= 5.5 or v >= 8.5: return WARNING
    if col == 'claytotal_r': return CRITICAL if v >= 50 else (WARNING if v >= 40 else NORMAL)
    if col == 'om_r':
        if v < 0.5: return CRITICAL
        if v < 1.0: return WARNING
        if v >= 3.0: return GOOD
    if col in ('cec7_r', 'ecec_r'): return CRITICAL if v < 6 else (WARNING if v < 10 else NORMAL)
    if col == 'dbovendry_r': return CRITICAL if v >= 1.75 else (WARNING if v >= 1.6 else NORMAL)
    return NORMAL

def format_val(val, decimals=1):
    if val is None or (isinstance(val, float) and math.isnan(val)): return '<span style="color:#ccc;">—</span>'
    return f"{val:.{decimals}f}"

# ── EN: Table Columns Definition / ES: Columnas de la tabla ──────
TABLE_COL_DEFS = [
    ('Hz / Horiz',          'hzname',        '',   0),
    ('Depth / Prof (cm)',   '__depth__',     '',   0),
    ('Sand / Arena %',      'sandtotal_r',   '',   1),
    ('Silt / Limo %',       'silttotal_r',   '',   1),
    ('Clay / Arcilla %',    'claytotal_r',   '🏔️',  1),
    ('Texture / Textura',   '__texture__',   '',   0),
    ('Dens. (g/cm³)',       'dbovendry_r',   '',   2),
    ('H₂O 10kPa (%)',       'wtenthbar_r',   '',   1),
    ('H₂O 33kPa (%)',       'wthirdbar_r',   '',   1),
    ('H₂O 1500kPa (%)',     'wfifteenbar_r', '',   1),
    ('OM / MO (%)',         'om_r',          '🌿',  2),
    ('pH (H₂O 1:1)',        'ph1to1h2o_r',   '⚗️',  2),
    ('EC / CE (dS/m)',      'ec_r',          '💧',  2),
    ('ECEC',                'ecec_r',        '',   1),
    ('ESP / PSI (%)',       'esp_r',         '🧂',  1),
    ('SAR',                 'sar_r',         '🧂',  1),
    ('CEC / CIC',           'cec7_r',        '',   1),
    ('Est. N (%)',          'n_estimado_r',  '',   3),
]

def build_header():
    ths = [f'<th style="padding:7px 9px; background:#f5f5f5; color:#444; border:1px solid #ddd; white-space:nowrap; font-size:11px; font-weight:600; text-align:center;">{badge} {label}</th>' for label, col, badge, _ in TABLE_COL_DEFS]
    return '<tr>' + ''.join(ths) + '</tr>'

def build_data_row(hz, bg):
    tds = []
    texture = get_texture_class(hz.get('sandtotal_r'), hz.get('silttotal_r'), hz.get('claytotal_r'))
    for label, col, badge, decs in TABLE_COL_DEFS:
        if col == '__depth__':
            top, bot = hz.get('hzdept_r'), hz.get('hzdepb_r')
            top_s = str(int(top)) if pd.notna(top) else '?'
            bot_s = str(int(bot)) if pd.notna(bot) else '?'
            tds.append(f'<td style="text-align:center; border:1px solid #e8e8e8; padding:5px 7px; white-space:nowrap; color:#555; font-size:11px;">{top_s}–{bot_s}</td>')
        elif col == '__texture__':
            color = TEXTURE_PALETTE.get(texture, '#aaa')
            tds.append(f'<td style="text-align:center; border:1px solid #e8e8e8; padding:5px 7px;"><span style="background:{color}18; color:{color}; border:1px solid {color}50; padding:2px 6px; border-radius:8px; font-size:10px; white-space:nowrap;">{texture}</span></td>')
        elif col == 'hzname':
            tds.append(f'<td style="padding:5px 9px; border:1px solid #e8e8e8; font-weight:600; color:#333; font-size:12px;">{hz.get(col, "—")}</td>')
        else:
            val = hz.get(col)
            style = get_alert_style(col, val) if col else 'text-align:center; color:#333;'
            tds.append(f'<td style="padding:5px 7px; border:1px solid #e8e8e8; font-size:11px; {style}">{format_val(val, decs)}</td>')
    return f'<tr style="background:{bg};">' + ''.join(tds) + '</tr>'

def build_wpavg_row(wp_dict):
    tds = []
    texture_w = get_texture_class(wp_dict.get('sandtotal_r_wpavg'), wp_dict.get('silttotal_r_wpavg'), wp_dict.get('claytotal_r_wpavg'))
    for label, col, badge, decs in TABLE_COL_DEFS:
        if col == '__depth__':
            tds.append(f'<td style="text-align:center; border:1px solid #c8d9e8; padding:6px 7px; font-size:10px; color:#567; font-style:italic;">0–{MAX_DEPTH_CM} cm</td>')
        elif col == '__texture__':
            color = TEXTURE_PALETTE.get(texture_w, '#aaa')
            tds.append(f'<td style="text-align:center; border:1px solid #c8d9e8; padding:6px 7px;"><span style="background:{color}18; color:{color}; border:1px solid {color}50; padding:2px 6px; border-radius:8px; font-size:10px;">{texture_w}</span></td>')
        elif col == 'hzname':
            tds.append(f'<td style="padding:6px 9px; border:1px solid #c8d9e8; font-weight:600; color:#456; font-size:11px; white-space:nowrap;">⚖️ Wt. Avg / Prom. (0–{MAX_DEPTH_CM}cm)</td>')
        else:
            wp_col = f'{col}_wpavg' if col and not col.startswith('__') else None
            val = wp_dict.get(wp_col) if wp_col else None
            base_style = get_alert_style(col, val) if col else 'text-align:center; color:#333;'
            base_style = base_style.replace('background:#ffe4e1', 'background:#ffe8e4').replace('background:#fff9e6', 'background:#fffae8').replace('background:#f0f9f0', 'background:#f0fbf0')
            tds.append(f'<td style="padding:6px 7px; border:1px solid #c8d9e8; font-size:11px; {base_style}">{format_val(val, decs)}</td>')
    return ('<tr style="background:#e8f1f8; border-top:2px solid #567;">' + ''.join(tds) + '</tr>')

# ── EN: Mukey mappings / ES: Mapa mukey → musym y área ──────────────────────────────
gdf_areas = gdf_clipped.copy()
gdf_areas['mukey'] = gdf_areas['mukey'].astype(str)
gdf_areas['area_ha'] = gdf_areas.geometry.to_crs(epsg=6933).area / 10_000
musym_map = gdf_areas.groupby('mukey')['musym'].first().to_dict()
area_map  = gdf_areas.groupby('mukey')['area_ha'].sum().to_dict()

# ── EN: HTML Legend / ES: Leyenda de alertas ──────────────────────
HTML_LEGEND = (
    '<div style="font-family:sans-serif; font-size:11px; margin:12px 0 18px; padding:10px 16px; background:#fafafa; border:1px solid #ddd; border-radius:6px; display:flex; flex-wrap:wrap; gap:8px; align-items:center;">'
    '<b style="color:#333; margin-right:5px;">🔔 Alerts / Alertas:</b>'
    '<span style="background:#ffe4e1; color:#8b0000; padding:2px 8px; border-radius:10px; font-weight:600; font-size:10px;">🔴 CRITICAL / CRÍTICO</span>'
    '<span style="background:#fff9e6; color:#b8860b; padding:2px 8px; border-radius:10px; font-weight:600; font-size:10px;">🟡 WARNING / ALERTA</span>'
    '<span style="background:#f0f9f0; color:#2d6a2d; padding:2px 8px; border-radius:10px; font-size:10px;">🟢 OPTIMAL / ÓPTIMO</span>'
    '<span style="color:#666; font-size:10px;">'
    '&nbsp;EC/CE ≥2 Warn / ≥4 Crit'
    '&nbsp;|&nbsp; ESP/PSI ≥6 Warn / ≥15 Crit'
    '&nbsp;|&nbsp; SAR ≥9 Warn / ≥13 Crit'
    '&nbsp;|&nbsp; pH &lt;5.5 or &gt;8.5 Warn'
    '&nbsp;|&nbsp; Clay/Arcilla ≥40% Warn'
    '&nbsp;|&nbsp; OM/MO &lt;1% Warn / &lt;0.5% Crit'
    '&nbsp;|&nbsp; Dens. ≥1.6 Warn / ≥1.75 Crit'
    '</span></div>'
)

# ── EN: Render Tables / ES: Renderizar tablas por MUKey ───────────────
if not df_horizons.empty:
    wpavg_lookup = df_wpavg.set_index('mukey').to_dict('index') if 'df_wpavg' in dir() and not df_wpavg.empty else {}

    html_all = f'<div style="font-family:sans-serif; max-width:100%; overflow-x:auto;">{HTML_LEGEND}'
    html_all += f'<p style="color:#555; font-size:12px; margin:0 0 12px;">📊 <b>{df_horizons["mukey"].nunique()} Map Units (MUKeys)</b>. Dominant component horizons + weighted average 0–{MAX_DEPTH_CM} cm.</p>'

    for mukey, df_mu in df_horizons.groupby('mukey'):
        mukey_str = str(mukey)
        musym   = musym_map.get(mukey_str, mukey_str)
        area_ha = area_map.get(mukey_str, 0)
        
        comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
        df_dom   = df_mu[df_mu['cokey'] == comp_dom].sort_values('hzdept_r').reset_index(drop=True)
        
        # 1. EN: Prepare the Series Name and UC Davis Link
        # 1. ES: Preparamos el nombre de la serie y el link a UC Davis
        compname = df_dom['compname'].iloc[0]
        comppct  = df_dom['comppct_r'].iloc[0]
        n_hz     = len(df_dom)
        
        import urllib.parse
        nombre_s_url = urllib.parse.quote(compname.lower())
        url_ficha = f"https://casoilresource.lawr.ucdavis.edu/sde/?series={nombre_s_url}#osd"

        # 2. EN: Build the header with the hyperlink
        # 2. ES: Construimos el encabezado con el hipervínculo
        html_all += (
            f'<details open style="margin-bottom:16px; border:1px solid #d0d0d0; border-radius:8px; overflow:hidden; box-shadow:0 1px 3px rgba(0,0,0,0.05);">'
            f'<summary style="background:linear-gradient(135deg,#f7f7f7,#e8e8e8); padding:11px 16px; font-weight:600; cursor:pointer; color:#333; font-size:13px; list-style:none; display:flex; flex-wrap:wrap; align-items:center; gap:10px;">'
            f'<span style="background:#fff; padding:3px 9px; border-radius:12px; font-size:12px; border:1px solid #ccc;">MUKey {mukey_str}</span>'
            f'<span>Unit/Unidad: <b>{musym}</b></span><span style="opacity:0.5;">|</span>'
            f'<span>'
            f'  <a href="{url_ficha}" target="_blank" style="color: #3498db; text-decoration: none; border-bottom: 1px dashed #3498db;">{compname}</a> 🔗'
            f'  <span style="opacity:0.6; font-weight:normal;">({comppct:.0f}% dom.)</span>'
            f'</span><span style="opacity:0.5;">|</span>'
            f'<span style="opacity:0.8;">{area_ha:.1f} ha &nbsp;· {n_hz} horiz.</span>'
            f'</summary><div style="padding:12px; background:#fff; overflow-x:auto;">'
            f'<table style="width:100%; border-collapse:collapse; font-size:12px;">'
            f'<thead>{build_header()}</thead><tbody>'
        )

        for i, (_, hz) in enumerate(df_dom.iterrows()):
            html_all += build_data_row(hz, '#fdfdfd' if i % 2 == 0 else '#ffffff')

        html_all += build_wpavg_row(wpavg_lookup.get(mukey_str, {}))
        html_all += '</tbody></table></div></details>'

    display(HTML(html_all + '</div>'))
else:
    print("⚠️ EN: df_horizons is empty. Check SDA connection. / ES: df_horizons vacío. Revisá la conexión al servidor SDA.")
    
    print(f"📋 df_horizons shape: {df_horizons.shape}")
print(f"📋 df_horizons vacío: {df_horizons.empty}")
if not df_horizons.empty:
    print(df_horizons.head(2))

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_lab_data_pro(pedon_key):
    url = f"https://ncsslabdatamart.sc.egov.usda.gov/rptExecute.aspx?p={pedon_key}&r=1"
    print(f"📡 EN: Data extracted from / ES: Datos extraídos de: {url}")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Connection": "keep-alive",
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        # --- Buscar TODAS las tablas ---
        all_tables = soup.find_all("table")
       # print(f"🔍 Tablas encontradas en la página: {len(all_tables)}")

        best_table = None
        best_col_count = 0

        for i, table in enumerate(all_tables):
            rows = table.find_all("tr")
            if not rows:
                continue
            # Contar columnas en la fila con más celdas
            max_cols = max(
                sum(int(td.get("colspan", 1)) for td in row.find_all(["td", "th"]))
                for row in rows
            )
            if max_cols > best_col_count:
                best_col_count = max_cols
                best_table = table

        if best_table is None:
            print("❌ No se encontró ninguna tabla.")
            return None

        #print(f"\n✅ Tabla seleccionada: {best_col_count} columnas")

        # --- Extraer filas respetando colspan/rowspan ---
        rows = best_table.find_all("tr")
        data = []
        for row in rows:
            cells = row.find_all(["td", "th"])
            row_data = []
            for cell in cells:
                # Limpiar el texto
                text = cell.get_text(separator=" ", strip=True)
                text = re.sub(r'\s+', ' ', text)
                # Expandir colspan para que cada columna tenga su valor
                colspan = int(cell.get("colspan", 1))
                row_data.extend([text] * colspan)
            if row_data:
                data.append(row_data)

        if not data:
            print("❌ La tabla está vacía.")
            return None

        # --- Detectar cuántas filas son encabezado ---
        # Estrategia: las primeras filas con <th> son el header
        header_rows = []
        data_rows = []
        for row in best_table.find_all("tr"):
            if row.find("th"):
                header_rows.append(row)
            else:
                data_rows.append(row)

        # Construir encabezados combinando filas de header
        def parse_row(row):
            cells = row.find_all(["td", "th"])
            result = []
            for cell in cells:
                text = re.sub(r'\s+', ' ', cell.get_text(separator=" ", strip=True))
                colspan = int(cell.get("colspan", 1))
                result.extend([text] * colspan)
            return result

        if header_rows:
            # Combinar múltiples filas de encabezado
            parsed_headers = [parse_row(r) for r in header_rows]
            max_len = max(len(h) for h in parsed_headers)
            # Rellenar para igualar longitud
            parsed_headers = [h + [''] * (max_len - len(h)) for h in parsed_headers]
            # Unir verticalmente: "Nivel1_Nivel2"
            combined_headers = []
            for col_idx in range(max_len):
                parts = []
                for row_h in parsed_headers:
                    val = row_h[col_idx] if col_idx < len(row_h) else ''
                    if val and val not in parts:
                        parts.append(val)
                combined_headers.append('_'.join(parts) if parts else f'col_{col_idx}')
        else:
            combined_headers = [f'col_{i}' for i in range(len(data[0]))]

        # --- Parsear filas de datos ---
        parsed_data = [parse_row(r) for r in data_rows]

        # Normalizar longitud de filas
        n_cols = len(combined_headers)
        normalized = []
        for row in parsed_data:
            if len(row) < n_cols:
                row += [''] * (n_cols - len(row))
            else:
                row = row[:n_cols]
            normalized.append(row)

        df = pd.DataFrame(normalized, columns=combined_headers)

        # --- Limpieza final ---
        # Eliminar filas completamente vacías o con solo espacios
        df = df.replace('', pd.NA)
        df = df.dropna(how='all')
        df = df.reset_index(drop=True)

        # Eliminar columnas completamente vacías
        df = df.dropna(axis=1, how='all')

        print(f"📐 DataFrame final: {df.shape[0]} filas × {df.shape[1]} columnas")
        return df

    except requests.exceptions.RequestException as e:
        print(f"❌ Error de red: {e}")
        return None
    except Exception as e:
        print(f"❌ Error inesperado: {e}")
        import traceback
        traceback.print_exc()
        return None

# --- EJECUCIÓN ---
PEDON_ID = 165
df_final = extract_lab_data_pro(PEDON_ID)

ruta_actual = os.getcwd()
df_lab_table = os.path.join(ruta_actual, f"pedon_{PEDON_ID}_LAB_CLEAN.csv")
df_final.to_csv(df_lab_table, index=False, encoding='utf-8-sig')
print(f"💾 Guardado exitosamente en: {df_lab_table}")
                            

In [ ]:
# ============================================================
# EN: CELL 5 — BLOCK 1: INTERACTIVE CHOROPLETH MAP (FOLIUM)
# ES: CELDA 5 — BLOQUE 1: MAPA COROPLETA INTERACTIVO (FOLIUM)
# ============================================================
from folium.features import GeoJsonTooltip
import folium
import branca.colormap as cm
import ipywidgets as widgets
from IPython.display import display, HTML

# ── EN: Variable Selector / ES: Selector de variable ──
# Asume que tu diccionario VARIABLES_MAP ahora se llama MAP_VARIABLES
MAP_OPTIONS = list(MAP_VARIABLES.keys()) + ['⚠️ Phosphorus / Fósforo (P) — N/A in SDA']

var_selector = widgets.Dropdown(
    options=MAP_OPTIONS,
    value=MAP_OPTIONS[0], # Uso el primer elemento por defecto en vez de hardcodear 'Arcilla'
    description='Variable:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='380px')
)

btn_generate = widgets.Button(
    description='🗺️ Generate Map',
    button_style='primary',
    layout=widgets.Layout(width='160px', margin='4px 8px')
)

out_map = widgets.Output()
current_map = None # Variable global para poder guardar el mapa en HTML al final

def generate_map(b=None):
    global current_map
    with out_map:
        out_map.clear_output(wait=True)
        selected_label = var_selector.value

        # ── EN: Special case for Phosphorus / ES: Caso especial para Fósforo ──
        if 'Fósforo' in selected_label or 'Phosphorus' in selected_label:
            display(HTML("""
            <div style="background:#fff3cd; border:2px solid #ffc107; border-radius:10px;
                        padding:24px 30px; max-width:680px; font-family:sans-serif; margin:12px 0;">
                <h3 style="color:#856404; margin:0 0 10px;">⚠️ EN: Phosphorus (P) Not Available / ES: Fósforo (P) No Disponible</h3>
                <p style="color:#555; margin:0 0 8px;">
                    <strong>EN:</strong> The USDA Soil Data Access (SDA) spatial database does not contain available Phosphorus (P) values for any horizon level.<br>
                    <strong>ES:</strong> La base de datos espacial USDA (SDA) no contiene valores de Fósforo disponible (P) para ningún nivel de horizonte.
                </p>
                <p style="color:#555; margin:0 0 8px;">
                    <strong>EN:</strong> P availability depends on specific mineralogy and fertilization history, hence it is excluded from generalized NRCS pedological models.<br>
                    <strong>ES:</strong> El Fósforo depende de la mineralogía específica y del historial de fertilización, por ello no se incluye en los modelos pedológicos.
                </p>
                <p style="color:#856404; font-weight:bold; margin:0;">
                    📋 Action / Acción: Realizar análisis de laboratorio o cargar datos locales.
                </p>
            </div>
            """))
            return

        data_col = MAP_VARIABLES[selected_label]
        col_wpavg = f'{data_col}_wpavg'

        if col_wpavg not in gdf_dash.columns:
            print(f"⚠️ EN: Column '{col_wpavg}' not found. Check previous cell. / ES: Columna '{col_wpavg}' no encontrada.")
            return

        gdf_plot = gdf_dash[['mukey', 'musym', 'geometry', col_wpavg]].copy()
        gdf_plot = gdf_plot.dropna(subset=[col_wpavg])

        if gdf_plot.empty:
            display(HTML("<div style='padding:16px; background:#f8d7da; border-radius:8px; color:#721c24;'>"
                         "⚠️ EN: No data available for this variable. / ES: No hay datos disponibles para esta variable.</div>"))
            return

        vmin = gdf_plot[col_wpavg].min()
        vmax = gdf_plot[col_wpavg].max()

        # ── EN: Create Folium map / ES: Crear mapa Folium ──
        m = folium.Map(
            location=centroid, # Variable actualizada de centro_lote a field_center
            zoom_start=14,
            tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
            attr='Google Satellite'
        )
        folium.TileLayer('CartoDB positron', name='Light Map / Mapa claro').add_to(m)

        # ── EN: Colormap selection / ES: Selección de paleta ──
        if data_col == 'ph1to1h2o_r':
            colormap = cm.LinearColormap(['#e74c3c','#e67e22','#27ae60','#f39c12','#c0392b'],
                                          vmin=vmin, vmax=vmax, caption=selected_label)
        elif data_col in ('ec_r', 'esp_r', 'sar_r'):
            colormap = cm.LinearColormap(['#27ae60','#f39c12','#e74c3c'], vmin=vmin, vmax=vmax, caption=selected_label)
        else:
            colormap = cm.LinearColormap(['#fffde7','#f9a825','#e65100'], vmin=vmin, vmax=vmax, caption=selected_label)

        colormap.add_to(m)

        # ── EN: Special note for estimated N / ES: Nota especial para N estimado ──
        is_estimated_n = data_col == 'n_estimado_r'

        def get_style(feature):
            val = feature['properties'].get(col_wpavg)
            color = colormap(val) if val is not None else '#CCCCCC'
            return {'fillColor': color, 'color': '#333', 'weight': 1.2, 'fillOpacity': 0.72}

        # ── EN: Custom Tooltip / ES: Tooltip personalizado ──
        n_note = " (Est: OM÷20)" if is_estimated_n else ""
        folium.GeoJson(
            gdf_plot.__geo_interface__,
            style_function=get_style,
            highlight_function=lambda x: {'weight': 3, 'fillOpacity': 0.95},
            tooltip=GeoJsonTooltip(
                fields=['musym', col_wpavg],
                aliases=['Unit / Unidad:', f'{selected_label}{n_note}:'],
                localize=True,
                sticky=True,
                style="font-family:sans-serif; font-size:13px;"
            ),
            name=selected_label
        ).add_to(m)

        # ── EN: Field boundary / ES: Contorno del lote ──
        folium.GeoJson(
            gdf_field.__geo_interface__, # Variable actualizada de gdf_lote a gdf_field
            style_function=lambda x: {'color': 'white', 'weight': 2.5, 'fillOpacity': 0},
            name='AOI Boundary / Límite del lote'
        ).add_to(m)

        folium.LayerControl().add_to(m)

        # ── EN: Map Title / ES: Título del mapa ──
        html_title = f"""
        <div style="position:fixed; top:10px; left:50%; transform:translateX(-50%);
                    background:white; padding:8px 18px; border-radius:8px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.25); font-family:sans-serif;
                    font-size:14px; font-weight:bold; z-index:9999; max-width:90%;">
            🗺️ {selected_label} &nbsp;·&nbsp; Wt. Avg / Prom. Ponderado 0–{MAX_DEPTH_CM} cm
            {'&nbsp;<span style="color:#e67e22;font-weight:bold;">⚠️ Est. from OM/MO</span>' if is_estimated_n else ''}
        </div>
        """
        m.get_root().html.add_child(folium.Element(html_title))

        current_map = m
        display(m)

        # ── EN: Summary Table / ES: Tabla resumen ──
        table_col_name = f'{selected_label} (0–{MAX_DEPTH_CM} cm)'
        html_summary = gdf_plot[['musym', col_wpavg]].rename(
            columns={'musym': 'Unit / Unidad', col_wpavg: table_col_name}
        ).style.format({table_col_name: '{:.2f}'}).set_table_styles(
            [{'selector': 'th', 'props': [('background', '#2c3e50'), ('color', 'white'), ('padding', '8px 14px')]},
             {'selector': 'td', 'props': [('padding', '6px 14px'), ('text-align', 'center')]}]
        ).to_html()
        
        display(HTML(f"<div style='margin-top:12px;'>{html_summary}</div>"))

btn_generate.on_click(generate_map)

control_panel = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0; font-family:sans-serif;'>🗺️ Block 1 — Choropleth Map / Bloque 1 — Mapa Coropleta</h3>"
                 "<p style='font-family:sans-serif; color:#555; font-size:13px; margin:0 0 10px;'>"
                 "EN: Select a variable to generate the weighted average map.<br>"
                 "ES: Seleccioná la variable y generá el mapa coropleta por promedio ponderado.</p>"),
    widgets.HBox([var_selector, btn_generate])
], layout=widgets.Layout(padding='14px', border='1px solid #bdc3c7', border_radius='8px', margin='8px 0'))

display(control_panel)
display(out_map)
generate_map()  # Generar con variable por defecto al ejecutar la celda

# ── EN: Save HTML map (Bug Fix) / ES: Guardar mapa en HTML (Bug Corregido) ──
def generate_map(b=None):
    global current_map1
    with out_map:
        # ... todo el código existente ...
        current_map1 = m
        display(m)
        
        # Mover el save acá adentro
        m.save('choropleth_map.html')
        print("✅ Map saved as 'choropleth_map.html'")

In [ ]:
# ============================================================
# CELDA 6 — BLOQUE 2: PERFILES EN PROFUNDIDAD (PLOTLY)
# ============================================================

VARIABLES_PERFIL = {
    'Textura (Arcilla / Limo / Arena)': ['claytotal_r', 'silttotal_r', 'sandtotal_r'],
    'pH':                    'ph1to1h2o_r',
    'CE — Salinidad (dS/m)': 'ec_r',
    'PSI — Sodicidad (%)':   'esp_r',
    'SAR':                   'sar_r',
    'Materia Orgánica (%)':  'om_r',
    'CIC (cmol/kg)':         'cec7_r',
    'N estimado (%) ⚠️':     'n_estimado_r',
    '⚠️ Fósforo (P) — No disponible en SDA': None,
}

COLORES_TEXTURA = {
    'claytotal_r': '#8B0000',
    'silttotal_r': '#C68642',
    'sandtotal_r': '#F4A460',
}
NOMBRES_TEXTURA = {
    'claytotal_r': 'Arcilla',
    'silttotal_r': 'Limo',
    'sandtotal_r': 'Arena',
}

# ── Obtener lista de unidades disponibles ──
mukeys_disp = df_horizons['mukey'].unique().tolist()
musym_map   = gdf_clipped[['mukey','musym']].drop_duplicates().set_index('mukey')['musym'].to_dict()

opciones_unidades = [
    f"Unidad {musym_map.get(str(mk), mk)} (mukey: {mk})"
    for mk in mukeys_disp
]

selector_var_p = widgets.Dropdown(
    options=list(VARIABLES_PERFIL.keys()),
    value='Textura (Arcilla / Limo / Arena)',
    description='Variable:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px')
)

selector_unidad = widgets.SelectMultiple(
    options=opciones_unidades,
    value=opciones_unidades[:min(3, len(opciones_unidades))],
    description='Unidades:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='360px', height='100px')
)

btn_perfil = widgets.Button(
    description='📊 Generar Perfiles',
    button_style='success',
    layout=widgets.Layout(width='180px', margin='4px 8px')
)

out_perfiles = widgets.Output()

def generar_perfiles(b=None):
    with out_perfiles:
        out_perfiles.clear_output(wait=True)
        label_sel = selector_var_p.value
        col_data  = VARIABLES_PERFIL[label_sel]

        # ── Fósforo: dato no disponible ──
        if col_data is None:
            display(HTML("""
            <div style="background:#fff3cd; border:2px solid #ffc107; border-radius:10px;
                        padding:24px 30px; max-width:680px; font-family:sans-serif; margin:12px 0;">
                <h3 style="color:#856404; margin:0 0 10px;">⚠️ Fósforo (P) — Dato No Disponible en SDA</h3>
                <p style="color:#555; margin:0;">
                    La base de datos Soil Data Access (USDA-NRCS) no incluye datos de Fósforo
                    en la tabla <code>chorizon</code> ni en ninguna otra tabla tabular.
                    Para incluir este dato, podés agregar manualmente tu análisis de laboratorio.
                </p>
            </div>
            """))
            return

        # Filtrar unidades seleccionadas
        mukeys_sel = []
        for opt in selector_unidad.value:
            mk = opt.split('mukey: ')[1].rstrip(')')
            mukeys_sel.append(mk)

        if not mukeys_sel:
            print("⚠️ Seleccioná al menos una unidad.")
            return

        n_unidades = len(mukeys_sel)
        es_textura    = isinstance(col_data, list)
        es_n_estimado = col_data == 'n_estimado_r'

        fig = make_subplots(
            rows=1, cols=n_unidades,
            subplot_titles=[
                f"Unidad {musym_map.get(mk, mk)}"
                for mk in mukeys_sel
            ],
            shared_yaxes=True
        )

        for col_idx, mk in enumerate(mukeys_sel, start=1):
            df_mu = df_horizons[df_horizons['mukey'].astype(str) == str(mk)]
            if df_mu.empty:
                continue

            # Componente dominante
            comp_dom = df_mu.sort_values('comppct_r', ascending=False)['cokey'].iloc[0]
            df_comp  = df_mu[df_mu['cokey'] == comp_dom].sort_values('hzdept_r')
            compname = df_comp['compname'].iloc[0]
            comppct  = df_comp['comppct_r'].iloc[0]

            if es_textura:
                # ── Barras horizontales apiladas por fracción textural ──
                for var_tex in col_data:
                    if var_tex not in df_comp.columns:
                        continue
                    valores_x = []
                    valores_y = []
                    nombres_hz = []
                    for _, hz in df_comp.iterrows():
                        top = hz['hzdept_r']
                        bot = hz['hzdepb_r']
                        val = hz[var_tex]
                        mid = (top + bot) / 2
                        esp = bot - top
                        valores_x.append(val if not pd.isna(val) else 0)
                        valores_y.append(mid)
                        nombres_hz.append(f"{hz['hzname']} ({top}–{bot} cm)")

                    show_legend = (col_idx == 1)
                    fig.add_trace(
                        go.Bar(
                            x=valores_x,
                            y=valores_y,
                            orientation='h',
                            name=NOMBRES_TEXTURA[var_tex],
                            marker_color=COLORES_TEXTURA[var_tex],
                            text=[f"{v:.0f}%" if v > 0 else '' for v in valores_x],
                            textposition='inside',
                            hovertemplate=f"<b>{NOMBRES_TEXTURA[var_tex]}</b><br>Valor: %{{x:.1f}}%<br>Profundidad: %{{y}} cm<extra></extra>",
                            legendgroup=NOMBRES_TEXTURA[var_tex],
                            showlegend=show_legend,
                            width=[hz['hzdepb_r'] - hz['hzdept_r'] for _, hz in df_comp.iterrows()]
                        ),
                        row=1, col=col_idx
                    )

            else:
                # ── Gráfico de área para variable escalar ──
                if col_data not in df_comp.columns:
                    continue

                # Construir línea de perfil: cada horizonte como escalón
                xs, ys = [], []
                for _, hz in df_comp.iterrows():
                    val = hz[col_data]
                    if pd.isna(val): val = None
                    xs.extend([val, val])
                    ys.extend([hz['hzdept_r'], hz['hzdepb_r']])

                color_linea = '#e67e22' if es_n_estimado else '#2980b9'
                color_fill  = 'rgba(230,126,34,0.2)' if es_n_estimado else 'rgba(41,128,185,0.15)'

                fig.add_trace(
                    go.Scatter(
                        x=xs,
                        y=ys,
                        mode='lines',
                        fill='tozerox',
                        fillcolor=color_fill,
                        line=dict(color=color_linea, width=2.5),
                        name=compname,
                        showlegend=(col_idx == 1),
                        hovertemplate=f"<b>{label_sel}</b><br>Valor: %{{x:.2f}}<br>Prof.: %{{y}} cm<extra></extra>"
                    ),
                    row=1, col=col_idx
                )

                # Línea de referencia 80 cm (zona de raíces)
                fig.add_hline(
                    y=PROF_MAX_CM, line_dash='dash', line_color='#27ae60',
                    annotation_text=f'{PROF_MAX_CM} cm (zona raíz)',
                    annotation_position='bottom right',
                    row=1, col=col_idx
                )

            # Subtítulo con componente dominante
            fig.layout.annotations[col_idx - 1]['text'] = (
                f"<b>Unidad {musym_map.get(mk, mk)}</b><br>"
                f"<span style='font-size:11px;color:#555;'>{compname} ({comppct:.0f}%)</span>"
            )

        # ── Layout general ──
        barmode = 'stack' if es_textura else None
        prof_max_plot = df_horizons['hzdepb_r'].max() if not df_horizons.empty else 200

        nota_title = " — <span style='color:#e67e22;'>⚠️ Estimado: N = MO ÷ 20</span>" if es_n_estimado else ""

        fig.update_layout(
            title=dict(
                text=f"📊 Perfil en Profundidad — {label_sel}{nota_title}",
                font=dict(size=16)
            ),
            barmode=barmode,
            height=560,
            template='plotly_white',
            legend=dict(orientation='h', y=-0.12, x=0.5, xanchor='center'),
            margin=dict(t=100, b=80)
        )

        # Eje Y invertido en todos los subplots (profundidad hacia abajo)
        for i in range(1, n_unidades + 1):
            yax = f'yaxis{i}' if i > 1 else 'yaxis'
            fig.update_layout(**{
                yax: dict(
                    autorange='reversed',
                    title='Profundidad (cm)' if i == 1 else '',
                    range=[prof_max_plot + 5, 0],
                    gridcolor='#eee'
                )
            })
            xax = f'xaxis{i}' if i > 1 else 'xaxis'
            x_title = '(%)' if es_textura else label_sel
            fig.update_layout(**{xax: dict(title=x_title, gridcolor='#eee')})

        fig.show()

        # Advertencia N estimado
        if es_n_estimado:
            display(HTML("""
            <div style="background:#fff3cd; border:1px solid #ffc107; border-radius:8px;
                        padding:12px 18px; font-family:sans-serif; font-size:13px; margin-top:8px; max-width:700px;">
                <b>⚠️ Nitrógeno estimado</b> — Los valores mostrados son una <em>aproximación teórica</em>
                calculada como <code>N = MO ÷ 20</code>, basada en la relación C:N de Bremner (1965).
                No reemplaza al análisis de laboratorio. La mineralización real depende de temperatura,
                humedad y actividad biológica.
            </div>
            """))

btn_perfil.on_click(generar_perfiles)

panel_perfiles = widgets.VBox([
    widgets.HTML("<h3 style='margin:8px 0; font-family:sans-serif;'>📊 Bloque 2 — Perfiles en Profundidad</h3>"
                 "<p style='font-family:sans-serif; color:#555; font-size:13px; margin:0 0 10px;'>Seleccioná variable y unidades para visualizar el perfil completo.</p>"),
    widgets.HBox([selector_var_p, selector_unidad, btn_perfil])
], layout=widgets.Layout(padding='14px', border='1px solid #bdc3c7', border_radius='8px', margin='8px 0'))

display(panel_perfiles)
display(out_perfiles)
generar_perfiles()  # Generar con valores por defecto al ejecutar la celda

In [ ]:
# ============================================================
# CONTOUR MAP — display inline in Jupyter cell output
# ============================================================
import json
import numpy as np
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
from shapely.geometry import LineString, mapping
from IPython.display import HTML, display
import folium

# ── Helpers ──────────────────────────────────────────────────
def latlon_to_tile(lat, lon, zoom):
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - np.log(np.tan(np.radians(lat)) + 1 / np.cos(np.radians(lat))) / np.pi) / 2 * n)
    return x, y

def tile_bounds(x, y, zoom):
    n = 2 ** zoom
    lon_min = x / n * 360 - 180
    lon_max = (x + 1) / n * 360 - 180
    lat_max = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * y / n))))
    lat_min = np.degrees(np.arctan(np.sinh(np.pi * (1 - 2 * (y + 1) / n))))
    return lon_min, lat_min, lon_max, lat_max

def terrarium_to_meters(r, g, b):
    return (r * 256 + g + b / 256) - 32768

# ── CRS y bbox ───────────────────────────────────────────────
gdf_wgs84    = gdf_field.to_crs('EPSG:4326')
centroid_lat = gdf_wgs84.geometry.centroid.iloc[0].y
centroid_lon = gdf_wgs84.geometry.centroid.iloc[0].x
minx, miny, maxx, maxy = gdf_wgs84.total_bounds

# ── Descargar tiles ──────────────────────────────────────────
zoom = 14
tile_min_x, tile_min_y = latlon_to_tile(maxy, minx, zoom)
tile_max_x, tile_max_y = latlon_to_tile(miny, maxx, zoom)

TILE_PX   = 256
tile_cols = tile_max_x - tile_min_x + 1
tile_rows = tile_max_y - tile_min_y + 1
mosaic    = np.zeros((tile_rows * TILE_PX, tile_cols * TILE_PX, 3), dtype=np.float32)

print(f'Descargando {tile_cols * tile_rows} tile(s)...')
for ty in range(tile_min_y, tile_max_y + 1):
    for tx in range(tile_min_x, tile_max_x + 1):
        url = f'https://s3.amazonaws.com/elevation-tiles-prod/terrarium/{zoom}/{tx}/{ty}.png'
        r   = requests.get(url)
        if r.status_code != 200:
            continue
        tile_img = np.array(Image.open(BytesIO(r.content)).convert('RGB'), dtype=np.float32)
        row_off  = (ty - tile_min_y) * TILE_PX
        col_off  = (tx - tile_min_x) * TILE_PX
        mosaic[row_off:row_off+TILE_PX, col_off:col_off+TILE_PX] = tile_img
      

elevation = terrarium_to_meters(mosaic[:,:,0], mosaic[:,:,1], mosaic[:,:,2])
print(f'DEM listo | {elevation.min():.1f} - {elevation.max():.1f} m')

# ── Bounds y conversion px -> latlon ─────────────────────────
geo_lon_min = tile_bounds(tile_min_x, tile_min_y, zoom)[0]
geo_lat_max = tile_bounds(tile_min_x, tile_min_y, zoom)[3]
geo_lon_max = tile_bounds(tile_max_x, tile_max_y, zoom)[2]
geo_lat_min = tile_bounds(tile_max_x, tile_max_y, zoom)[1]
total_px_w  = tile_cols * TILE_PX
total_px_h  = tile_rows * TILE_PX

def pixel_to_latlon(px, py):
    lon = geo_lon_min + (px / total_px_w) * (geo_lon_max - geo_lon_min)
    lat = geo_lat_max - (py / total_px_h) * (geo_lat_max - geo_lat_min)
    return lon, lat

# ── Extraer contornos ────────────────────────────────────────
fig_tmp, ax_tmp = plt.subplots()
cs = ax_tmp.contour(elevation, levels=15)
plt.close(fig_tmp)

contour_features = []
for i, level_segs in enumerate(cs.allsegs):
    level_value = cs.levels[i]
    for seg in level_segs:
        if len(seg) < 2:
            continue
        coords_geo = [pixel_to_latlon(px, py) for px, py in seg]
        contour_features.append({
            'type': 'Feature',
            'geometry': mapping(LineString(coords_geo)),
            'properties': {'elevation_m': round(float(level_value), 1)}
        })
print(f'{len(contour_features)} segmentos de curvas generados')

# ── Colores ──────────────────────────────────────────────────
m = folium.Map(location=[centroid_lat, centroid_lon], zoom_start=15,
               tiles="Esri.WorldImagery", attr="Esri")

elev_values = [f['properties']['elevation_m'] for f in contour_features]
elev_min, elev_max = min(elev_values), max(elev_values)
cmap = plt.cm.get_cmap('RdYlGn')

def elev_to_hex(val):
    norm = (val - elev_min) / (elev_max - elev_min + 1e-9)
    r, g, b, _ = cmap(norm)
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

for feat in contour_features:
    feat['properties']['color'] = elev_to_hex(feat['properties']['elevation_m'])

contour_geojson = json.dumps({'type': 'FeatureCollection', 'features': contour_features})
field_geojson   = json.dumps(gdf_wgs84.__geo_interface__)

# ── Leyenda ──────────────────────────────────────────────────
n_stops    = 8
stops      = []
ticks_html = ''
for i in range(n_stops + 1):
    t = i / n_stops
    r, g, b, _ = cmap(t)
    hex_c = '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))
    stops.append('{} {}%'.format(hex_c, int(t * 100)))
    if i % 2 == 0:
        label = '{:.0f} m'.format(elev_min + t * (elev_max - elev_min))
        ticks_html += (
            '<span style="position:absolute;left:{}%;transform:translateX(-50%);'
            'font-size:10px;color:#aaa;top:16px;white-space:nowrap;">{}</span>'
        ).format(int(t * 100), label)

gradient_css = ', '.join(stops)

# ── HTML (fragmento, sin DOCTYPE ni body) ────────────────────
# CLAVE: position:absolute en vez de fixed, contenedor con height explicito
html = '''
<div style="position:relative;width:100%;height:580px;
            border-radius:12px;overflow:hidden;
            font-family:Segoe UI,Arial,sans-serif;">

  <!-- Tooltip elevacion -->
  <div id="tt" style="display:none;
       position:absolute;top:14px;left:50%;
       transform:translateX(-50%);
       background:rgba(0,0,0,0.82);color:#fff;
       padding:7px 18px;border-radius:20px;
       font-size:14px;font-weight:600;
       z-index:1000;pointer-events:none;
       border:1px solid rgba(255,255,255,0.2);"></div>

  <!-- Leyenda -->
  <div style="position:absolute;bottom:20px;left:50%;
       transform:translateX(-50%);
       background:rgba(15,15,30,0.88);
       border:1px solid rgba(255,255,255,0.12);
       padding:10px 18px 30px 18px;
       border-radius:10px;z-index:1000;min-width:300px;">
    <div style="font-size:10px;text-transform:uppercase;
                letter-spacing:1.5px;color:#aaa;margin-bottom:6px;">Elevacion</div>
    <div style="height:13px;border-radius:5px;
                background:linear-gradient(to right,''' + gradient_css + ''');
                border:1px solid rgba(255,255,255,0.1);"></div>
    <div style="position:relative;height:28px;">''' + ticks_html + '''</div>
  </div>

  <!-- Info -->
  <div style="position:absolute;top:14px;right:14px;
       background:rgba(15,15,30,0.85);color:#aaa;
       font-size:11px;padding:8px 12px;
       border-radius:8px;z-index:1000;line-height:1.8;
       border:1px solid rgba(255,255,255,0.1);">
    Hover = elevacion | Scroll = zoom | Drag = mover
  </div>

  <!-- Mapa -->
  <div id="leafmap" style="width:100%;height:100%;"></div>
</div>

<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
(function() {
  var CONTOURS = ''' + contour_geojson + ''';
  var FIELD    = ''' + field_geojson + ''';

  var map = L.map("leafmap").setView([''' + str(centroid_lat) + ''', ''' + str(centroid_lon) + '''], 15);

  L.tileLayer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    {attribution: "Esri", maxZoom: 19}
  ).addTo(map);

  var tt = document.getElementById("tt");

  // Campo
  L.geoJSON(FIELD, {
    style: {color: "#00ff88", weight: 2.5, fillOpacity: 0.07, fillColor: "#00ff88"}
  }).addTo(map);

  // Curvas con hover
  L.geoJSON(CONTOURS, {
    style: function(feat) {
      return {color: feat.properties.color, weight: 2.8, opacity: 0.6};
    },
    onEachFeature: function(feat, layer) {
      var elev = feat.properties.elevation_m;
      layer.on("mouseover", function() {
        this.setStyle({weight: 4, opacity: 1, color: "#ffffff"});
        tt.textContent = "Elevacion: " + elev + " m";
        tt.style.display = "block";
      });
      layer.on("mouseout", function() {
        this.setStyle({weight: 0.8, opacity: 0.6, color: feat.properties.color});
        tt.style.display = "none";
      });
    }
  }).addTo(map);
})();
</script>
'''

display(HTML(html))
# ── Guardar ─────────────────────────────────────────────────
import webbrowser, os

html_file = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>body{{margin:0;padding:0;background:#0f0f1e;}}</style>
</head><body>
{html}
</body></html>"""

with open("contour_layer_map.html", "w", encoding="utf-8") as f:
    f.write(html_file)
#para abrir automativcamente en un html
#webbrowser.open("file://" + os.path.abspath("contour_layer_map.html"))
print("✅ Listo")
print('Mapa generado correctamente')


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# 1. Preparación de los ejes (Lon/Lat)
x_coords = np.linspace(geo_lon_min, geo_lon_max, elevation.shape[1])
y_coords = np.linspace(geo_lat_min, geo_lat_max, elevation.shape[0])

# 2. Factor de exageración vertical
exag_vertical = 1.5 

# 3. Creación de la superficie 3D (Cambiamos 'terrain' por 'earth')
fig_3d = go.Figure(data=[go.Surface(
    z=elevation * exag_vertical, 
    x=x_coords, 
    y=y_coords,
    colorscale='earth',  # <--- Cambiado aquí
    colorbar=dict(title='Elevación (m)', thickness=20),
    hovertemplate='Lon: %{x:.4f}<br>Lat: %{y:.4f}<br>Alt: %{z:.1f}m<extra></extra>'
)])

# 4. Configuración del Layout
fig_3d.update_layout(
    title='Análisis Topográfico 3D',
    scene=dict(
        xaxis_title='Longitud',
        yaxis_title='Latitud',
        zaxis_title='Elevación (m)',
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=0.3), 
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    template="plotly_dark"
)

display(fig_3d)

In [ ]:
import numpy as np
import plotly.graph_objects as go
import requests
from PIL import Image
from io import BytesIO

# 1. Función para descargar el mosaico de imágenes (Textura)
def get_base_map_texture(tile_min_x, tile_max_x, tile_min_y, tile_max_y, zoom):
    t_cols = tile_max_x - tile_min_x + 1
    t_rows = tile_max_y - tile_min_y + 1
    # Esri usa 256x256 px por tile
    texture_mosaic = np.zeros((t_rows * 256, t_cols * 256, 3), dtype=np.uint8)

    print(f"📥 Descargando {t_cols * t_rows} tiles de Esri World Imagery...")
    for ty in range(tile_min_y, tile_max_y + 1):
        for tx in range(tile_min_x, tile_max_x + 1):
            # URL de Esri World Imagery (fíjate que el orden de x/y puede variar según el servidor)
            url = f"https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{zoom}/{ty}/{tx}"
            r = requests.get(url)
            if r.status_code == 200:
                tile_img = np.array(Image.open(BytesIO(r.content)).convert('RGB'))
                r_off = (ty - tile_min_y) * 256
                c_off = (tx - tile_min_x) * 256
                texture_mosaic[r_off:r_off+256, c_off:c_off+256] = tile_img
    return texture_mosaic

# 2. Generar la textura (Usando tus variables previas)
texture_img = get_base_map_texture(tile_min_x, tile_max_x, tile_min_y, tile_max_y, zoom)

# 1. Preparar la malla (aplanar coordenadas y convertir colores a formato Plotly)
X, Y = np.meshgrid(x_coords, y_coords[::-1]) # Invertimos Y para que el Norte quede arriba
x_f, y_f, z_f = X.flatten(), Y.flatten(), (elevation * 1.5).flatten()

# Convertir la imagen de Esri a una lista de strings 'rgb(r,g,b)'
color_strings = [f'rgb({r},{g},{b})' for r, g, b in texture_img.reshape(-1, 3)]

# 2. Generar los triángulos de la malla (2 por cada celda del grid)
rows, cols = elevation.shape
i_idx, j_idx, k_idx = [], [], []
for r in range(rows - 1):
    for c in range(cols - 1):
        v1, v2 = r * cols + c, r * cols + (c + 1)
        v3, v4 = (r + 1) * cols + c, (r + 1) * cols + (c + 1)
        i_idx.extend([v1, v2]); j_idx.extend([v2, v4]); k_idx.extend([v3, v3])

# 3. Crear el gráfico usando Mesh3d
fig_3d_base = go.Figure(data=[go.Mesh3d(
    x=x_f, y=y_f, z=z_f,
    i=i_idx, j=j_idx, k=k_idx,
    customdata=elevation.flatten(),
    vertexcolor=color_strings, # Aquí aplicamos la foto real
    lighting=dict(ambient=0.6, diffuse=0.8, roughness=0.9, specular=0.1),
    hoverinfo='none',
    hovertemplate='<b>Elevación Real:</b> %{customdata:.2f} m<extra></extra>'
)])

# 4. Ajustar el Layout
fig_3d_base.update_layout(
    title='Maqueta Digital 3D (Textura Real Esri World Imagery)',
    scene=dict(
        xaxis_visible=True, yaxis_visible=True,
        zaxis=dict(title='Elevación (m)', backgroundcolor="rgb(20, 20, 30)"),
        aspectratio=dict(x=1, y=1, z=0.1),
        bgcolor="rgb(10, 10, 15)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    template="plotly_dark"
)
import webbrowser, os
fig_3d_base.write_html("3DDEM.html")
webbrowser.open("file://" + os.path.abspath("3DDEM.html"))
#fig_3d_base.show()